# Inventory Order Recommendation Engine

This notebook demonstrates:
- Sales velocity calculation
- ABC classification using Pareto distribution
- Rule-based ordering logic

All user parameters are directly after import statements.

In [1]:
import numpy as np
import pandas as pd
import datetime
from IPython.display import display, HTML
from pathlib import Path

In [2]:
# ============================
# USER DEFINED PARAMETERS (edit here).
# ============================

# ----------------------------
# Note: example data is only valid from 2025-12-31 to 2026-02-04.
# ----------------------------

period_start = pd.to_datetime("2025-12-31")
period_end = pd.to_datetime("2026-02-03")
delivery_date = pd.to_datetime("2026-02-09")
next_delivery_date = pd.to_datetime("2026-02-23")

# ----------------------------
# Derived metrics calculated automatically.
# ----------------------------

period_days = (period_end - period_start).days + 1
lead_time = (delivery_date - period_end).days
days_between_deliveries = (next_delivery_date - delivery_date).days

# Flag reporting periods shorter than 14 days (shown on final output). 
# Short periods produce unreliable sales velocity and ordering recommendations.
if period_days >= 14:
    planning_status = "Valid planning window (Period Days >= 14 days)"
else:
    planning_status = "Window too short for planning (Period Days < 14 days)"

# ----------------------------
# Validation.
# ----------------------------

# Validate reporting period spans at least one day.
if period_days < 1:
    raise ValueError(
        f"Invalid reporting period: period_start ({period_start}) must be on or before period_end ({period_end})."
    )

# ----------------------------
# Output (for inspection).
# ----------------------------

print(f"Period Days = {period_days}")
print(f"Lead Time = {lead_time}")
print(f"Days between deliveries = {days_between_deliveries}")
print(f"Planning Status = {planning_status}")

Period Days = 35
Lead Time = 6
Days between deliveries = 14
Planning Status = Valid planning window (Period Days >= 14 days)


In [3]:
# ============================
# Read csv files.
# ============================

filepath = Path("..") / "data"

# Use encoding="latin1" because the source CSV files are not UTF-8 encoded, which is common with Excel/Windows exports.
# product_id is a key used to join tables and must remain a string to preserve integrity.
read_args = {"encoding": "latin1",
            "dtype": {"product_id": str}}

def load_csv(name):
    return pd.read_csv(filepath / f"{name}.csv", **read_args)

tblProduct = load_csv("tblProduct")
tblSales = load_csv("tblSales")
tblPurchases = load_csv("tblPurchases")
tblInventorySnapshot = load_csv("tblInventorySnapshot")

In [4]:
# ============================
# Stylize dataframe output.
# ============================

def df_styler(df):
    styled_df = df.style

    # ----------------------------
    # Adjust display headers
    # ----------------------------
    header_map = {
        "abc_class": "ABC<br>Class",
        "beg_qty": "Beg<br>Qty",
        "cum_pct_of_sales": "Cumulative<br>Percent<br>of Sales",
        "current_days_of_inventory": "Current<br>Days of<br>Inventory",
        "current_depletion_date": "Current<br>Depletion Date",
        "daily_velocity": "Avg<br>Daily<br>Sales",
        "end_qty": "End<br>Qty",
        "inventory_at_delivery": "Inventory<br>at Delivery",
        "inventory_post_delivery": "Inventory<br>Post Delivery",
        "pct_of_sales": "Percent<br>of Sales",
        "product_id": "ProdID",
        "product_name": "Product",
        "projected_days_of_inventory": "Projected<br>Days of<br>Inventory",
        "projected_depletion_date": "Projected<br>Depletion Date",
        "purchase_unit_size": "Purchase Unit<br>Size",
        "purchase_units_to_order": "Purchase Units<br>to Order",
        "purchased": "Purchased",
        "sold": "Sold",
        "target_days_of_inventory_post_delivery": "Target Days<br>of Inventory<br>Post Delivery",
        "target_inventory_post_delivery": "Target Inventory<br>Post Delivery",
        "units_to_order": "Units<br>to Order"
    }

    display_headers = [header_map.get(col, col) for col in df.columns]

    styled_df = styled_df.relabel_index(display_headers, axis=1)
    
    # ----------------------------
    # Apply display formatting
    # ----------------------------
    display_formats = {
        "period_start": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else "",
        "period_end": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else "",
        "daily_velocity": "{:.2f}",
        "pct_of_sales": "{:.2%}",
        "cum_pct_of_sales": "{:.2%}",
        "current_depletion_date": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else "",
        "projected_depletion_date": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else ""
    }

    applicable_formats = {
        col: fmt for col, fmt in display_formats.items() if col in df.columns
    }

    if applicable_formats:
        styled_df = styled_df.format(applicable_formats)

    # ----------------------------
    # Base table styling
    # ----------------------------
    styled_df = styled_df.set_table_styles(
        [
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("font-family", "Arial, Helvetica, sans-serif"),
                    ("font-size", "12px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("padding", "6px 8px"),
                    ("vertical-align", "bottom"),
                    ("white-space", "normal")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("padding", "6px 8px")
                ]
            }
        ],
        overwrite=False
    )

    # ----------------------------
    # Group columns by datatype: text or non-text 
    # ----------------------------
    text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    non_text_cols = [col for col in df.columns if col not in text_cols]

    # ----------------------------
    # Align column headers by datatype
    # ----------------------------
    header_styles = []

    for col in text_cols:
        col_idx = df.columns.get_loc(col)
        header_styles.append({
            "selector": f"th.col_heading.level0.col{col_idx}",
            "props": [("text-align", "left")]
        })

    for col in non_text_cols:
        col_idx = df.columns.get_loc(col)
        header_styles.append({
            "selector": f"th.col_heading.level0.col{col_idx}",
            "props": [("text-align", "right")]
        })

    if header_styles:
        styled_df = styled_df.set_table_styles(header_styles, overwrite=False)

    # ----------------------------
    # Align column values by datatype
    # ----------------------------
    if text_cols:
        styled_df = styled_df.set_properties(
            subset=text_cols,
            **{"text-align": "left"}
        )

    if non_text_cols:
        styled_df = styled_df.set_properties(
            subset=non_text_cols,
            **{"text-align": "right"}
        )

    # ----------------------------
    # Highlight ABC rows
    # ----------------------------
    if "abc_class" in df.columns:

        def highlight_abc_rows(row):
            val = str(row["abc_class"])

            if val.startswith("A"):
                return ["background-color: #D4EDDA"] * len(row)
            elif val.startswith("B"):
                return ["background-color: #FFF3CD"] * len(row)
            elif val.startswith("C"):
                return ["background-color: #F8D7DA"] * len(row)

            return [""] * len(row)

        styled_df = styled_df.apply(highlight_abc_rows, axis=1)

    return styled_df

In [5]:
# ============================
# Build base table.
# ============================

# Build base table by combining the product master (tblProduct)
# with beginning inventory (tblInventorySnapshot),
# purchases (tblPurchases), and sales (tblSales).

# ----------------------------
# Extract beginning inventory from tblInventorySnapshot
# ----------------------------

# Convert snapshot_date to datetime for accurate filtering.
tblInventorySnapshot["snapshot_date"] = pd.to_datetime(tblInventorySnapshot["snapshot_date"])

# SELECT product_id, qty_on_hand AS beg_qty
# FROM tblInventorySnapshot
# WHERE snapshot_date = period_start

beg_inventory = (
    tblInventorySnapshot.loc[
        tblInventorySnapshot["snapshot_date"] == period_start,
        ["product_id", "qty_on_hand"]
    ]
    .rename(columns={"qty_on_hand": "beg_qty"})
)

# ----------------------------
# Extract purchases from tblPurchases
# ----------------------------

# Convert received_date to datetime for accurate filtering.
tblPurchases["received_date"] = pd.to_datetime(tblPurchases["received_date"])

# SELECT product_id, SUM(total_qty_purchased) AS purchased
# FROM tblPurchases
# WHERE received_date BETWEEN period_start AND period_end
# GROUP BY product_id

purchases = (
    tblPurchases.loc[
        tblPurchases["received_date"].between(period_start, period_end), 
        ["product_id", "total_qty_purchased"]
    ]
    .groupby("product_id", as_index=False)["total_qty_purchased"].sum()
    .rename(columns={"total_qty_purchased": "purchased"})
)

# ----------------------------
# Extract sales from tblSales
# ----------------------------

# Convert sale_date to datetime for accurate filtering.
tblSales["sale_date"] = pd.to_datetime(tblSales["sale_date"])

# SELECT product_id, SUM(quantity) AS sold
# FROM tblSales
# WHERE sale_date BETWEEN period_start AND period_end
# GROUP BY product_id

sales = (
    tblSales.loc[
        tblSales["sale_date"].between(period_start, period_end),
        ["product_id", "quantity"]
    ]
    .groupby("product_id", as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "sold"})
)

# ----------------------------
# Build base table
# ----------------------------

# SELECT 
#   p.product_id, 
#   p.product_name, 
#   p.purchase_unit_size,
#   COALESCE(b.beg_qty, 0) AS beg_qty,
#   COALESCE(pr.purchased, 0) AS purchased,
#   COALESCE(s.sold, 0) AS sold
# FROM tblProduct p
# LEFT JOIN beg_inventory b 
#   ON p.product_id = b.product_id
# LEFT JOIN purchases pr
#   ON p.product_id = pr.product_id
# LEFT JOIN sales s
#   ON p.product_id = s.product_id

base = (
    tblProduct[["product_id", "product_name", "purchase_unit_size"]]
    .merge(beg_inventory, on="product_id", how="left")
    .merge(purchases, on="product_id", how="left")
    .merge(sales, on="product_id", how="left")
    .fillna({
        "beg_qty": 0,
        "purchased": 0,
        "sold": 0
    })
)

# ----------------------------
# Cast numeric columns to integers
# ----------------------------

# Left joins introduce missing values for products with no inventory, purchases, or sales.
# Pandas stores missing numeric values as NaN, which causes these columns to become floats.
# Cast back to integers.

base[["beg_qty", "purchased", "sold"]] = (
    base[["beg_qty", "purchased", "sold"]].astype(int)
)

# ----------------------------
# Calculate ending inventory quantity
# ----------------------------

base["end_qty"] = base["beg_qty"] + base["purchased"] - base["sold"]

# ----------------------------
# Output (for inspection)
# ----------------------------

display(df_styler(base.head()))


,ProdID,Product,Purchase UnitSize,BegQty,Purchased,Sold,EndQty
0,1088638,Protein Bar,24,30,72,79,23
1,1253167,Lay's Classic Potato Chips,24,12,96,62,46
2,1331817,Planters Peanuts,24,30,0,24,6
3,1711217,Ruffles Cheddar & Sour Cream,24,26,96,87,35
4,2012714,Wheat Thins,24,14,0,10,4


In [6]:
# ============================
# Extend base to create sales_metrics.
# ============================

# Include:
#  - Daily sales velocity
#  - Current days of inventory
#  - Current depletion date

# Assumption:
# Daily sales velocity remains constant when estimating
# current days of inventory and depletion date.

sales_metrics = base.copy()

# ----------------------------
# Daily sales velocity
# ----------------------------

sales_metrics["daily_velocity"] = sales_metrics["sold"] / period_days

# ----------------------------
# Current days of inventory
# ----------------------------

sales_metrics["current_days_of_inventory"] = (
    sales_metrics["end_qty"]
    / sales_metrics["daily_velocity"].replace(0, np.nan)
)

# Floor current days of inventory to whole days.
# Cast to pandas nullable Int64 for integer representation while retaining missing values (<NA>).
# Note: NumPy int64 (lowercase 'i') cannot be used because it does not support null values.
sales_metrics["current_days_of_inventory"] = (
    np.floor(sales_metrics["current_days_of_inventory"])
).astype("Int64")

# ----------------------------
# Current depletion date
# ----------------------------

sales_metrics["current_depletion_date"] = (
    period_end + pd.to_timedelta(sales_metrics["current_days_of_inventory"], unit="D")
)

# ----------------------------
# Output (for inspection)
# ----------------------------

display_columns = [
    "product_name",
    "beg_qty",
    "purchased",
    "sold",
    "end_qty",
    "daily_velocity",
    "current_days_of_inventory",
    "current_depletion_date"
]
display(df_styler(sales_metrics[display_columns].head()))

,Product,BegQty,Purchased,Sold,EndQty,AvgDailySales,CurrentDays ofInventory,CurrentDepletion Date
0,Protein Bar,30,72,79,23,2.26,10,2026-02-13
1,Lay's Classic Potato Chips,12,96,62,46,1.77,25,2026-02-28
2,Planters Peanuts,30,0,24,6,0.69,8,2026-02-11
3,Ruffles Cheddar & Sour Cream,26,96,87,35,2.49,14,2026-02-17
4,Wheat Thins,14,0,10,4,0.29,14,2026-02-17


In [7]:
# ============================
# Extend sales_metrics to create abc_classification using Pareto-based cumulative percent of sold.
# ============================

# Include:
#  - ABC Classification
#  - Ordering policy based on ABC class

abc_classification = sales_metrics.copy()

# ----------------------------
# ABC Classification
# ----------------------------

# Before calculating cumulative percent of units sold, sort by sold descending.
# product_id is used as a secondary sort key to break ties consistently.
abc_classification = abc_classification.sort_values(
    by=["sold", "product_id"],
    ascending=[False, True]
).reset_index(drop=True)

# Calculate each product's share of total units sold in the reporting period.
# Stop execution if total_sold == 0.
total_sold = abc_classification["sold"].sum()
if total_sold == 0:
    raise ValueError(
        "Total units sold is 0 for the reporting period. Unable to calculate ABC classification."
    )
abc_classification["pct_of_sales"] = abc_classification["sold"] / total_sold

# Calculate cumulative percent of units sold.
abc_classification["cum_pct_of_sales"] = abc_classification["pct_of_sales"].cumsum()

# Products contributing to the first 80% of cumulative percent of units sold are Class A.
# Products contributing to the next 15% are Class B.
# The remaining products are Class C.
abc_classification["abc_class"] = np.select(
    [
        abc_classification["cum_pct_of_sales"] <= 0.80,
        abc_classification["cum_pct_of_sales"] <= 0.95
    ],
    ["A", "B"],
    default="C"
)

# ----------------------------
# Ordering policy based on ABC class
# ----------------------------

# Orders for each class include enough post-delivery inventory 
# to last from the upcoming delivery to the following delivery, 
# plus a class-specific buffer:
# A: +14 days of extra inventory
# B: +7 days of extra inventory
# C: no additional days of extra inventory
abc_policy_map = {
    "A": {"target_days": days_between_deliveries + 14},
    "B": {"target_days": days_between_deliveries + 7},
    "C": {"target_days": days_between_deliveries}
}

# Target days of inventory after delivery.
abc_classification["target_days_of_inventory_post_delivery"] = (
    abc_classification["abc_class"].map(lambda x: abc_policy_map[x]["target_days"])
)

# ----------------------------
# Output (for inspection)
# ----------------------------

display_columns = [
    "product_name",
    "sold",
    "pct_of_sales",
    "cum_pct_of_sales",
    "abc_class",
    "target_days_of_inventory_post_delivery"
]
display(df_styler(abc_classification[display_columns]))

,Product,Sold,Percentof Sales,CumulativePercentof Sales,ABCClass,Target Daysof InventoryPost Delivery
0,Sour Patch Kids,90,10.99%,10.99%,A,28
1,Ruffles Cheddar & Sour Cream,87,10.62%,21.61%,A,28
2,Protein Bar,79,9.65%,31.26%,A,28
3,Ritz Crackers Snack Pack,68,8.30%,39.56%,A,28
4,Lay's Classic Potato Chips,62,7.57%,47.13%,A,28
5,Trail Mix,52,6.35%,53.48%,A,28
6,Nerds,49,5.98%,59.46%,A,28
7,Snickers,48,5.86%,65.32%,A,28
8,Skittles,38,4.64%,69.96%,A,28
9,Pringles Original,34,4.15%,74.11%,A,28


In [8]:
# ============================
# Extend abc_classification to create order_recommendations.
# ============================

# Include:
#  - Projected inventory at delivery (before incoming shipment arrives)
#  - Target inventory after delivery
#  - Units needed to meet target inventory
#  - Convert units to purchase units (cases, packs, etc.)

order_recommendations = abc_classification.copy()

# ----------------------------
# Projected inventory at delivery (before incoming shipment arrives)
# ----------------------------

order_recommendations["inventory_at_delivery"] = (
    np.maximum(
        order_recommendations["end_qty"] - np.ceil(order_recommendations["daily_velocity"] * lead_time),
        0
    ).astype(int)
)
    
# ----------------------------
# Target inventory after delivery
# ----------------------------

order_recommendations["target_inventory_post_delivery"] = np.ceil(
    order_recommendations["target_days_of_inventory_post_delivery"]
    * order_recommendations["daily_velocity"]
).astype(int)

# ----------------------------
# Units needed to meet target inventory
# ----------------------------

order_recommendations["units_to_order"] = np.maximum(
    order_recommendations["target_inventory_post_delivery"]
    - order_recommendations["inventory_at_delivery"],
    0
).astype(int)

# ----------------------------
# Convert units to purchase units (cases, packs, etc.)
# ----------------------------

# Round up to ensure enough inventory is ordered.
# Return <NA> when purchase_unit_size is missing or zero.
order_recommendations["purchase_units_to_order"] = (
    np.ceil(
        order_recommendations["units_to_order"]
        / order_recommendations["purchase_unit_size"].replace(0, np.nan)
    ).astype("Int64")
)

# ----------------------------
# Output (for inspection)
# ----------------------------

display_columns = [
    "product_name",
    "abc_class",
    "inventory_at_delivery",
    "target_inventory_post_delivery",
    "units_to_order",
    "purchase_unit_size",
    "purchase_units_to_order"
]
display(df_styler(order_recommendations[display_columns]))

,Product,ABCClass,Inventoryat Delivery,Target InventoryPost Delivery,Unitsto Order,Purchase UnitSize,Purchase Unitsto Order
0,Sour Patch Kids,A,34,72,38,28,2
1,Ruffles Cheddar & Sour Cream,A,20,70,50,24,3
2,Protein Bar,A,9,64,55,24,3
3,Ritz Crackers Snack Pack,A,8,55,47,24,2
4,Lay's Classic Potato Chips,A,35,50,15,24,1
5,Trail Mix,A,2,42,40,24,2
6,Nerds,A,69,40,0,28,0
7,Snickers,A,12,39,27,28,1
8,Skittles,A,18,31,13,28,1
9,Pringles Original,A,21,28,7,24,1


In [9]:
# ============================
# Extend order_recommendations to create post_delivery_projection.
# ============================

# Include:
#  - Projected inventory on hand immediately after delivery
#  - Projected days inventory will last after delivery
#  - Projected depletion date after delivery

post_delivery_projection = order_recommendations.copy()

# ----------------------------
# Projected inventory on hand immediately after delivery
# ----------------------------

post_delivery_projection["inventory_post_delivery"] = (
    post_delivery_projection["inventory_at_delivery"]
    + (
        post_delivery_projection["purchase_units_to_order"] 
        * post_delivery_projection["purchase_unit_size"]
    )
)

# ----------------------------
# Projected days inventory will last after delivery
# ----------------------------

post_delivery_projection["projected_days_of_inventory"] = (
    np.floor(
        post_delivery_projection["inventory_post_delivery"]
        / post_delivery_projection["daily_velocity"].replace(0, np.nan)
    )
).astype("Int64")

# ----------------------------
# Projected depletion date after delivery
# ----------------------------

post_delivery_projection["projected_depletion_date"] = (
    delivery_date 
    + pd.to_timedelta(post_delivery_projection["projected_days_of_inventory"], unit="D")
)

# If no order is placed, retain the current depletion date rather than
# re-anchoring the depletion date to delivery_date.
post_delivery_projection.loc[
    post_delivery_projection["purchase_units_to_order"] == 0,
    "projected_depletion_date"
] = post_delivery_projection["current_depletion_date"]

# ----------------------------
# Output (for inspection)
# ----------------------------

display_columns = [
    "product_name",
    "inventory_at_delivery",
    "purchase_units_to_order",
    "inventory_post_delivery",
    "current_depletion_date",
    "projected_days_of_inventory",
    "projected_depletion_date"
]
display(df_styler(post_delivery_projection[display_columns]))

,Product,Inventoryat Delivery,Purchase Unitsto Order,InventoryPost Delivery,CurrentDepletion Date,ProjectedDays ofInventory,ProjectedDepletion Date
0,Sour Patch Kids,34,2,90,2026-02-22,35,2026-03-16
1,Ruffles Cheddar & Sour Cream,20,3,92,2026-02-17,37,2026-03-18
2,Protein Bar,9,3,81,2026-02-13,35,2026-03-16
3,Ritz Crackers Snack Pack,8,2,56,2026-02-13,28,2026-03-09
4,Lay's Classic Potato Chips,35,1,59,2026-02-28,33,2026-03-14
5,Trail Mix,2,2,50,2026-02-10,33,2026-03-14
6,Nerds,69,0,69,2026-03-30,49,2026-03-30
7,Snickers,12,1,40,2026-02-18,29,2026-03-10
8,Skittles,18,1,46,2026-02-26,42,2026-03-23
9,Pringles Original,21,1,45,2026-03-02,46,2026-03-27


In [10]:
# ============================
# Build and display final business reports:
# Products to Order and Products to Review.
# ============================

# ----------------------------
# Build report header with period metrics
# ----------------------------

page_header_html = f"""
<h1 style="text-align:left; font-family:Arial;">Inventory Order Recommendation Report</h1>
<p style="text-align:left; font-family:Arial;">
    <strong>Report Run Date:</strong> {datetime.date.today().strftime('%Y-%m-%d')}<br>
    <strong>Report Period Start:</strong> {period_start.strftime('%Y-%m-%d')}<br>
    <strong>Report Period End:</strong> {period_end.strftime('%Y-%m-%d')}<br>
    <strong>Planning Status:</strong> {planning_status}<br>
    <strong>Delivery Date:</strong> {delivery_date.strftime('%Y-%m-%d')}<br>
    <strong>Next Delivery Date:</strong> {next_delivery_date.strftime('%Y-%m-%d')}<br>
</p>
<hr style="border:2px solid #333;">
"""

# ----------------------------
# Build Products to Order
# ----------------------------

products_to_order = post_delivery_projection.loc[
    post_delivery_projection["purchase_units_to_order"] != 0
]

display_columns = [
    "product_id",
    "product_name",
    "abc_class",
    "daily_velocity",
    "end_qty",
    "current_depletion_date",
    "purchase_units_to_order",
    "inventory_post_delivery",
    "projected_depletion_date"
]

products_to_order_styled = (
    df_styler(products_to_order[display_columns])
    .hide(axis="index")
)

# ----------------------------
# Build Products to Review
# ----------------------------

products_to_review = post_delivery_projection.loc[
    post_delivery_projection["abc_class"] == "C"
]

display_columns = [
    "product_id",
    "product_name",
    "abc_class",
    "daily_velocity",
    "end_qty",
    "current_depletion_date"
]

products_to_review_styled = (
    df_styler(products_to_review[display_columns])
    .hide(axis="index")
)

# ----------------------------
# Display final report
# ----------------------------

display(HTML(page_header_html))
display(HTML("<h2>Products to Order</h2>"))
display(products_to_order_styled)
display(HTML("<h2>Products to Review</h2>"))
display(products_to_review_styled)


ProdID,Product,ABCClass,AvgDailySales,EndQty,CurrentDepletion Date,Purchase Unitsto Order,InventoryPost Delivery,ProjectedDepletion Date
9292315,Sour Patch Kids,A,2.57,50,2026-02-22,2,90,2026-03-16
1711217,Ruffles Cheddar & Sour Cream,A,2.49,35,2026-02-17,3,92,2026-03-18
1088638,Protein Bar,A,2.26,23,2026-02-13,3,81,2026-03-16
2732499,Ritz Crackers Snack Pack,A,1.94,20,2026-02-13,2,56,2026-03-09
1253167,Lay's Classic Potato Chips,A,1.77,46,2026-02-28,1,59,2026-03-14
7023463,Trail Mix,A,1.49,11,2026-02-10,2,50,2026-03-14
7165581,Snickers,A,1.37,21,2026-02-18,1,40,2026-03-10
6334728,Skittles,A,1.09,25,2026-02-26,1,46,2026-03-23
6825224,Pringles Original,A,0.97,27,2026-03-02,1,45,2026-03-27
8306905,Haribo Goldbears,A,0.89,18,2026-02-23,1,40,2026-03-26


ProdID,Product,ABCClass,AvgDailySales,EndQty,CurrentDepletion Date
8551457,Snyder's Pretzels,C,0.26,12,2026-03-21
2858379,Kit Kat,C,0.20,7,2026-03-10
2424707,Jelly Belly Jelly Beans,C,0.17,22,2026-06-11
7944400,Tootsie Roll,C,0.17,16,2026-05-07
9044404,Granola Bar,C,0.14,14,2026-05-12
8860153,Mike and Ike,C,0.11,3,2026-03-01
2982488,Popcorn (Microwave Bag),C,0.09,14,2026-07-16
6341525,Goldfish Crackers,C,0.06,5,2026-05-01
9115955,Laffy Taffy,C,0.06,13,2026-09-18
6230104,Hershey's Milk Chocolate Bar,C,0.03,22,2028-03-14


In [11]:
# ============================
# Export report as HTML.
# ============================

# ----------------------------
# Compile HTML
# ----------------------------

# HTML metadata to set print landscape as default.
print_landscape_html = (
f"""
<style>
    @media print {{
        @page {{
            size: landscape;
            margin: 0.5in;
        }}
    }}
</style>
"""
)

full_html = (
    print_landscape_html + 
    page_header_html +
    "<h2 style='font-family:Arial;'>Products to Order</h2>" +
    products_to_order_styled.to_html() +
    "<h2 style='font-family:Arial; margin-top:40px;'>Products to Review</h2>" +
    products_to_review_styled.to_html()
)

# ----------------------------
# Save as HTML file
# ----------------------------
with open("Inventory_Order_Recommendation_Report.html", "w", encoding="utf-8") as f:
    f.write(full_html)

print("✅ Report exported successfully!")
print("📄 File saved as: Inventory_Order_Recommendation_Report.html")
print("   → Open the file in your browser → Ctrl+P → Save as PDF")

✅ Report exported successfully!
📄 File saved as: Inventory_Order_Recommendation_Report.html
   → Open the file in your browser → Ctrl+P → Save as PDF
